In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold', 'axes.titlesize': 12})
os.makedirs('figures', exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f'figures/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# Upload cesnet_institution_hourly.csv when prompted
if not os.path.exists('cesnet_institution_hourly.csv'):
    from google.colab import files
    files.upload()

df = pd.read_csv('cesnet_institution_hourly.csv', parse_dates=['time'])
df.head(10)

In [ ]:
ts = df.set_index('time').drop(columns='id_time')
diffs = ts.index.to_series().diff().dropna()

print(f'Rows                 : {len(ts):,}')
print(f'Columns              : {ts.shape[1]}')
print(f'Start                : {ts.index.min()}')
print(f'End                  : {ts.index.max()}')
print(f'Frequency            : {diffs.mode()[0]} (regular: {diffs.nunique() == 1})')
print(f'Duplicate timestamps : {ts.index.duplicated().sum()}')
print(f'At least 500 rows    : {len(ts) >= 500}')
ts.describe().T[['count','mean','std','min','50%','max']].round(2)

In [ ]:
gaps = ts[ts['n_bytes'].isna()]
print('Hours with no data:', len(gaps), f'({len(gaps)/len(ts):.3%})')
print(gaps.index.tolist())

fig, ax = plt.subplots(2, 1, figsize=(11, 5), gridspec_kw={'height_ratios': [1, 2]})
ax[0].fill_between(ts.index, 0, ts['n_bytes'].isna().astype(int), step='mid', color='crimson')
ax[0].set_ylim(0, 1.2); ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(['present', 'missing'])
ax[0].set_title('Missing hours along the timeline')

cnt = ts.isna().sum().sort_values(ascending=False).head(10)
sns.barplot(x=cnt.values, y=cnt.index, ax=ax[1], color='steelblue')
ax[1].set_title('Missing values per column (top 10)'); ax[1].set_xlabel('Count')
save('01_missing_values')

In [ ]:
# Gap-free copy for plots that need a continuous series (time-based interpolation)
ts_f = ts.asfreq('h').interpolate(method='time')
y = ts_f['n_bytes'] / 1e9          # target: GB per hour
y.name = 'GB per hour'
print(len(y), 'points,', y.isna().sum(), 'missing')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(y.index, y, color='steelblue', lw=0.5, alpha=0.6, label='Hourly')
ax.plot(y.index, y.rolling(24).mean(), color='darkorange', lw=1.2, label='24-hour mean')
ax.plot(y.index, y.rolling(24*7).mean(), color='black', lw=1.8, label='7-day mean')
ax.set_title('Hourly traffic volume of one institution (CESNET3 network)')
ax.set_ylabel('GB per hour'); ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
save('02_line_plot')

In [ ]:
zoom = y['2023-11-06':'2023-12-03']
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(zoom.index, zoom, color='steelblue', lw=1)
ax.set_title('Four-week zoom: daily and weekly rhythm'); ax.set_ylabel('GB per hour')
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
save('03_zoom_four_weeks')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for a, (col, c, lab) in zip(axes, [('n_flows','teal','Flows per hour'),
                                   ('n_packets','purple','Packets per hour'),
                                   ('n_bytes','steelblue','Bytes per hour')]):q1, q3 = y.quantile([.25, .75]); iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
out = y[(y < lo) | (y > hi)]
print(f'Mean {y.mean():.1f} | median {y.median():.1f} | std {y.std():.1f} | skew {y.skew():.2f} | kurtosis {y.kurt():.2f}  (GB/h)')
print(f'IQR rule: {len(out)} outlier hours ({len(out)/len(y):.2%}), upper fence {hi:.1f} GB/h')
print('Top 5 busiest hours:'); print(y.nlargest(5).round(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(y, bins=60, ax=ax[0], color='steelblue'); ax[0].set_title('Histogram of hourly volume'); ax[0].set_xlabel('GB per hour')
sns.boxplot(x=y, ax=ax[1], color='lightsteelblue'); ax[1].set_title('Box plot (outliers = bursts)'); ax[1].set_xlabel('GB per hour')
save('05_distribution')
    a.plot(ts_f.index, ts_f[col], color=c, lw=0.5); a.set_ylabel(lab)
axes[0].set_title('Volume metrics side by side')
save('04_volume_metrics')

In [ ]:
q1, q3 = y.quantile([.25, .75]); iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
out = y[(y < lo) | (y > hi)]
print(f'Mean {y.mean():.1f} | median {y.median():.1f} | std {y.std():.1f} | skew {y.skew():.2f} | kurtosis {y.kurt():.2f}  (GB/h)')
print(f'IQR rule: {len(out)} outlier hours ({len(out)/len(y):.2%}), upper fence {hi:.1f} GB/h')
print('Top 5 busiest hours:'); print(y.nlargest(5).round(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(y, bins=60, ax=ax[0], color='steelblue'); ax[0].set_title('Histogram of hourly volume'); ax[0].set_xlabel('GB per hour')
sns.boxplot(x=y, ax=ax[1], color='lightsteelblue'); ax[1].set_title('Box plot (outliers = bursts)'); ax[1].set_xlabel('GB per hour')
save('05_distribution')

In [ ]:
dec24 = seasonal_decompose(y, model='additive', period=24, extrapolate_trend='freq')
fig = dec24.plot(); fig.set_size_inches(12, 8)
fig.suptitle('Additive decomposition, period = 24 hours', y=1.01, fontweight='bold')
save('06_decomposition_daily')

seasonal_strength = max(0, 1 - dec24.resid.var() / (dec24.resid + dec24.seasonal).var())
trend_strength    = max(0, 1 - dec24.resid.var() / (dec24.resid + dec24.trend).var())
print(f'Daily seasonal strength: {seasonal_strength:.2f} | Trend strength: {trend_strength:.2f}  (0 = none, 1 = very strong)')

In [ ]:
dec168 = seasonal_decompose(y, model='additive', period=168, extrapolate_trend='freq')
fig = dec168.plot(); fig.set_size_inches(12, 8)
fig.suptitle('Additive decomposition, period = 168 hours (weekly)', y=1.01, fontweight='bold')
save('07_decomposition_weekly')

weekly_strength = max(0, 1 - dec168.resid.var() / (dec168.resid + dec168.seasonal).var())
print(f'Weekly seasonal strength: {weekly_strength:.2f}')

In [ ]:
# Czech local time makes daily behaviour easier to interpret than UTC
y_local = y.copy()
y_local.index = y_local.index.tz_convert('Europe/Prague')
names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
hour = y_local.groupby(y_local.index.hour).mean()
dow  = y_local.groupby(y_local.index.dayofweek).mean()
month = y_local.groupby([y_local.index.year, y_local.index.month]).mean()
month.index = [f'{a}-{b:02d}' for a, b in month.index]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot(hour.index, hour.values, marker='o', color='steelblue')
ax[0].set_title('Average by hour of day (Prague time)'); ax[0].set_xlabel('Hour'); ax[0].set_ylabel('GB per hour')
ax[1].bar(names, dow.values, color='steelblue'); ax[1].set_title('Average by day of week')
ax[2].bar(month.index, month.values, color='steelblue'); ax[2].set_title('Average by month'); ax[2].tick_params(axis='x', rotation=60)
save('08_seasonal_profiles')
print('Busiest hour :', hour.idxmax(), f'({hour.max():.1f} GB/h)')
print('Quietest hour:', hour.idxmin(), f'({hour.min():.1f} GB/h)')

In [ ]:
heat = y_local.groupby([y_local.index.dayofweek, y_local.index.hour]).mean().unstack()
heat.index = names
plt.figure(figsize=(12, 3.6))
sns.heatmap(heat, cmap='YlOrRd', cbar_kws={'label': 'GB per hour'})
plt.title('Average traffic: weekday vs hour (Prague time)'); plt.xlabel('Hour of day'); plt.ylabel('')
save('09_heatmap_weekday_hour')

In [ ]:
# Czech public holidays inside the dataset period (from weekends_and_holidays.csv)
holidays = pd.to_datetime(['2023-11-17','2023-12-25','2023-12-26','2024-01-01','2024-03-29',
                           '2024-04-01','2024-05-01','2024-05-08','2024-07-05'])
local_date = y_local.index.tz_localize(None).normalize()
day_type = np.where(local_date.isin(holidays), 'Holiday',
            np.where(local_date.dayofweek >= 5, 'Weekend', 'Working day'))
cmp = pd.DataFrame({'GB per hour': y_local.values, 'Day type': day_type})

print(cmp.groupby('Day type')['GB per hour'].agg(['count','mean','median','std']).round(1))
plt.figure(figsize=(7, 4))
sns.boxplot(data=cmp, x='Day type', y='GB per hour', order=['Working day','Weekend','Holiday'], color='lightsteelblue')
plt.title('Traffic by day type')
save('10_day_type_boxplot')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
plot_acf(y, lags=24*8, ax=ax[0]); ax[0].set_title('ACF up to 8 days')
plot_pacf(y, lags=72, ax=ax[1], method='ywm'); ax[1].set_title('PACF up to 72 hours')
save('11_acf_pacf')

In [ ]:
cols = ['n_flows','n_packets','n_bytes','sum_n_dest_ip','sum_n_dest_asn','sum_n_dest_ports',
        'tcp_udp_ratio_bytes','dir_ratio_bytes','avg_duration','avg_ttl']
plt.figure(figsize=(8, 6.5))
sns.heatmap(ts_f[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 8})
plt.title('Correlation between traffic metrics')
save('12_correlation')

In [ ]:
adf_stat, adf_p = adfuller(y)[:2]
print(f'ADF statistic = {adf_stat:.2f}, p-value = {adf_p:.2e}')

In [ ]:
print('DATASET')
print(f'  Period : {ts.index.min():%d %b %Y %H:%M} to {ts.index.max():%d %b %Y %H:%M} UTC')
print(f'  Rows   : {len(ts):,} | Columns: {ts.shape[1]} | Missing hours: {len(gaps)} ({len(gaps)/len(ts):.3%})')
print('\nTARGET: n_bytes (GB per hour)')
print(f'  mean {y.mean():.1f} | median {y.median():.1f} | std {y.std():.1f} | min {y.min():.1f} | max {y.max():.1f}')
print(f'  IQR outlier hours: {len(out)} ({len(out)/len(y):.2%})')
print('\nPATTERNS')
print(f'  Daily seasonal strength : {seasonal_strength:.2f}')
print(f'  Weekly seasonal strength: {weekly_strength:.2f}')
print(f'  Trend strength          : {trend_strength:.2f}')
print(f'  Busiest hour  (Prague)  : {hour.idxmax()}:00 ({hour.max():.1f} GB/h)')
print(f'  Quietest hour (Prague)  : {hour.idxmin()}:00 ({hour.min():.1f} GB/h)')
print(f'  Busiest weekday: {names[int(dow.idxmax())]} | Quietest: {names[int(dow.idxmin())]}')
print('  Mean GB/h by day type  :', cmp.groupby('Day type')['GB per hour'].mean().round(1).to_dict())

In [ ]:
import shutil
shutil.make_archive('milestone1_figures', 'zip', 'figures')
from google.colab import files
files.download('milestone1_figures.zip')

In [ ]:
MILESTONE 2

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold', 'axes.titlesize': 12})
os.makedirs('figures', exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f'figures/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

if not os.path.exists('cesnet_institution_hourly.csv'):
    from google.colab import files
    files.upload()          # choose cesnet_institution_hourly.csv when prompted

df = pd.read_csv('cesnet_institution_hourly.csv', parse_dates=['time'])
ts = df.set_index('time').drop(columns='id_time')
ts_f = ts.asfreq('h').interpolate(method='time')
y = ts_f['n_bytes'] / 1e9
y.name = 'GB per hour'
print(len(y), 'points | missing:', y.isna().sum(), f'| mean {y.mean():.1f} GB/h')

In [ ]:
!pip install -q prophet autots tsfresh

In [ ]:
import logging, shutil
from statsmodels.tsa.statespace.varmax import VARMAX
logging.getLogger('cmdstanpy').setLevel(logging.ERROR); logging.getLogger('prophet').setLevel(logging.ERROR)

# Test set = last 7 days (168 h). Fitting window W = last 8 weeks of the training data
H = 168
train, test = y.iloc[:-H], y.iloc[-H:]
W = train.iloc[-24*7*8:]
full = pd.concat([W, test])

def metrics(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float); e = a - p
    return dict(RMSE=np.sqrt(np.mean(e**2)), MAE=np.mean(np.abs(e)), MAPE=np.mean(np.abs(e / a)) * 100)

fc168, fc1 = {}, {}     # forecasts by model: Task A (7 days ahead), Task B (1 hour ahead)
print(f'Train {len(train)} pts | Test {H} h ({test.index.min():%d %b %H:%M} -> {test.index.max():%d %b %H:%M} UTC) | fitting window {len(W)} pts')
print(f'Reference: the test week has mean {test.mean():.1f} and std {test.std():.1f} GB/h. '
      f'Always predicting the training mean would give RMSE = {metrics(test, np.repeat(W.mean(), H))["RMSE"]:.1f}.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
sd = train.diff(24).dropna()
plot_acf(train, lags=24*4, ax=ax[0]); ax[0].set_title('ACF of the series (4 days): slow decay + 24 h bumps')
plot_acf(sd, lags=24*7, ax=ax[1]); ax[1].set_title('ACF after seasonal differencing (lag 24)')
save('m2_01_acf')

In [ ]:
def stationarity(s, name):
    s = s.dropna()
    adf_p = adfuller(s, autolag='AIC')[1]
    kpss_p = kpss(s, regression='c', nlags='auto')[1]
    verdict = 'stationary' if (adf_p < 0.05 and kpss_p >= 0.05) else \
              'non-stationary' if (adf_p >= 0.05 and kpss_p < 0.05) else 'conflicting'
    return dict(series=name, ADF_p=adf_p, KPSS_p=kpss_p, verdict=verdict)

st = pd.DataFrame([stationarity(y, 'Original'), stationarity(y.diff(1), 'd=1'),
                   stationarity(y.diff(24), 'D=1 (lag 24)'), stationarity(y.diff(168), 'D=1 (lag 168)'),
                   stationarity(y.diff(24).diff(1), 'D=1 (24) + d=1')]).set_index('series')
st['ADF_p'] = st['ADF_p'].map('{:.2e}'.format); st['KPSS_p'] = st['KPSS_p'].map('{:.3f}'.format)
st

In [ ]:
def stationarity(s, name):
    s = s.dropna()
    adf_p = adfuller(s, autolag='AIC')[1]
    kpss_p = kpss(s, regression='c', nlags='auto')[1]
    verdict = 'stationary' if (adf_p < 0.05 and kpss_p >= 0.05) else \
              'non-stationary' if (adf_p >= 0.05 and kpss_p < 0.05) else 'conflicting'
    return dict(series=name, ADF_p=adf_p, KPSS_p=kpss_p, verdict=verdict)

st = pd.DataFrame([stationarity(y, 'Original'), stationarity(y.diff(1), 'd=1'),
                   stationarity(y.diff(24), 'D=1 (lag 24)'), stationarity(y.diff(168), 'D=1 (lag 168)'),
                   stationarity(y.diff(24).diff(1), 'D=1 (24) + d=1')]).set_index('series')
st['ADF_p'] = st['ADF_p'].map('{:.2e}'.format); st['KPSS_p'] = st['KPSS_p'].map('{:.3f}'.format)
st

**Conclusion.** The raw series is *conflicting* (ADF says stationary, KPSS says not): the slowly moving level (trend strength 0.53) breaks the constant-mean assumption. One regular difference (`d=1`) fixes this, so ARIMA uses `d=1`. A seasonal difference at lag 24 (`D=1, m=24`) also gives a clean result, which supports the seasonal part of SARIMA. AR, MA and ARMA are fitted on the original series with a constant term and are kept as members of the family.

## 3. Parameter selection and justification
* **ACF / PACF** suggest the order: a PACF cut-off points to the AR order `p`, an ACF cut-off to the MA order `q`.
* Candidates are then compared with **AIC** (and BIC), which reward fit and penalise complexity. All candidates use the same window `W`, so the numbers are comparable.

In [ ]:
def aic_of(order, seasonal=(0,0,0,0)):
    m = SARIMAX(W, order=order, seasonal_order=seasonal, trend='c' if order[1] == 0 and seasonal[1] == 0 else 'n',
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
    return m.aic, m.bic

ar_tab   = pd.DataFrame({p: aic_of((p,0,0)) for p in [1,2,3,4,6,12,24]}, index=['AIC','BIC']).T
ma_tab   = pd.DataFrame({q: aic_of((0,0,q)) for q in [1,2,3,4,6]}, index=['AIC','BIC']).T
arma_tab = pd.DataFrame({(p,q): aic_of((p,0,q)) for p in [1,2,3] for q in [1,2,3]}, index=['AIC','BIC']).T
p_best = int(ar_tab['AIC'].idxmin()); q_best = int(ma_tab['AIC'].idxmin()); pq_best = tuple(int(v) for v in arma_tab['AIC'].idxmin())
print('AR(p)   -> best p =', p_best); print(ar_tab.round(0).T.to_string())
print('\nMA(q)   -> best q =', q_best); print(ma_tab.round(0).T.to_string())
print('\nARMA    -> best (p,q) =', pq_best); print(arma_tab.round(0).sort_values('AIC').head(4).to_string())

In [ ]:
# Seasonal candidates (m = 24: the daily cycle). A small grid keeps it fast enough for Colab.
grid = [((1,0,0),(1,1,1,24)), ((1,0,1),(1,1,1,24)), ((2,0,1),(1,1,1,24)), ((2,0,2),(0,1,1,24)), ((1,0,1),(0,1,1,24))]
sel = pd.DataFrame([dict(order=o, seasonal_order=s, **dict(zip(['AIC','BIC'], aic_of(o, s)))) for o, s in grid]).sort_values('AIC').reset_index(drop=True)
s_order, s_seasonal = sel.loc[0, 'order'], sel.loc[0, 'seasonal_order']
sarima_name = f'SARIMA{s_order}{s_seasonal}'
print('Best seasonal model by AIC:', sarima_name)
sel

## 4. Classical models
Two forecasting tasks are judged for every model, because they answer two real questions:
* **Task A – 7-day-ahead (capacity planning):** the model sees only the training data and forecasts all 168 hours at once.
* **Task B – 1-hour-ahead (operations and burst alerts):** parameters are learned on training data only; then, hour by hour through the test week, the model predicts the next hour after seeing the real traffic so far.

**Exogenous variables (ARIMAX / SARIMAX)** must be known in advance to be usable, so only *calendar* information is used: hour-of-day and weekly waves, a weekend flag and a Czech public-holiday flag. Same-hour packets or flows are **not** used (unknown at forecast time = data leakage).

In [ ]:
hol = pd.to_datetime(['2023-11-17','2023-12-25','2023-12-26','2024-01-01','2024-03-29','2024-04-01','2024-05-01','2024-05-08','2024-07-05'])
def calendar(idx):
    loc = idx.tz_convert('Europe/Prague'); hr = loc.hour.values; wk = (loc.dayofweek * 24 + loc.hour).values
    c = {}
    for k in (1, 2, 3): c[f'day_sin{k}'] = np.sin(2*np.pi*k*hr/24); c[f'day_cos{k}'] = np.cos(2*np.pi*k*hr/24)
    for k in (1, 2):    c[f'wk_sin{k}']  = np.sin(2*np.pi*k*wk/168); c[f'wk_cos{k}'] = np.cos(2*np.pi*k*wk/168)
    c['weekend'] = (loc.dayofweek >= 5).astype(float)
    c['holiday'] = loc.tz_localize(None).normalize().isin(hol).astype(float)
    return pd.DataFrame(c, index=idx)

fits = {}
def run(name, order, seasonal=(0,0,0,0), exog=False):
    t0 = time.time()
    m = SARIMAX(W, exog=calendar(W.index) if exog else None, order=order, seasonal_order=seasonal,
                trend='c' if order[1] == 0 and seasonal[1] == 0 else 'n',
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
    fc = m.get_forecast(H, exog=calendar(test.index) if exog else None)                    # Task A
    fc168[name] = pd.Series(fc.predicted_mean.values, index=test.index)
    a = m.apply(full, exog=calendar(full.index) if exog else None)                          # Task B (parameters fixed)
    fc1[name] = pd.Series(a.get_prediction(start=len(W)).predicted_mean.values, index=test.index)
    fits[name] = (m, fc)
    print(f'{name:36s} AIC {m.aic:7.0f} | A RMSE {metrics(test, fc168[name])["RMSE"]:6.2f} | B RMSE {metrics(test, fc1[name])["RMSE"]:6.2f} | {time.time()-t0:.0f}s')

### 4.1 AR(p) – today's traffic is predicted from its own recent past

In [ ]:
run(f'AR({p_best})', (p_best, 0, 0))

### 4.2 MA(q) – predicted from the recent forecast errors (shocks)

In [ ]:
run(f'MA({q_best})', (0, 0, q_best))

### 4.3 ARMA(p,q) – both ideas together

In [ ]:
run(f'ARMA{pq_best}', (pq_best[0], 0, pq_best[1]))

### 4.4 ARIMA(p,1,q) – ARMA after one regular difference (`d=1`, justified by the KPSS result)

In [ ]:
run(f'ARIMA({pq_best[0]},1,{pq_best[1]})', (pq_best[0], 1, pq_best[1]))
arima_name = f'ARIMA({pq_best[0]},1,{pq_best[1]})'
print(fits[arima_name][0].summary().tables[1])

### 4.5 SARIMA – adds the 24-hour seasonal part (order chosen by AIC in Section 3)

In [ ]:
run(sarima_name, s_order, s_seasonal)
print(fits[sarima_name][0].summary().tables[1])

run(f'ARIMAX({pq_best[0]},1,{pq_best[1]})', (pq_best[0], 1, pq_best[1]), exog=True)
run(f'SARIMAX{s_order}{s_seasonal}', s_order, s_seasonal, exog=True)

In [ ]:
run(f'ARIMAX({pq_best[0]},1,{pq_best[1]})', (pq_best[0], 1, pq_best[1]), exog=True)
run(f'SARIMAX{s_order}{s_seasonal}', s_order, s_seasonal, exog=True)

### 4.7 Holt-Winters (triple exponential smoothing)
Level + a 24-hour seasonal pattern, both updated every hour with weights α and γ. Additive seasonality because the daily swing is roughly constant in size; no trend term because the trend is a slow drift, not sustained growth. Task B re-uses the weights learned on the training data while the state keeps updating through the test week.

In [ ]:
hw = ExponentialSmoothing(W, trend=None, seasonal='add', seasonal_periods=24, initialization_method='estimated').fit()
hw_name = 'Holt-Winters (m=24)'
fc168[hw_name] = pd.Series(hw.forecast(H).values, index=test.index)
al, ga = hw.params['smoothing_level'], hw.params['smoothing_seasonal']
hw1 = ExponentialSmoothing(full, trend=None, seasonal='add', seasonal_periods=24, initialization_method='known',
                           initial_level=hw.params['initial_level'], initial_seasonal=hw.params['initial_seasons']
                           ).fit(smoothing_level=al, smoothing_seasonal=ga, optimized=False)
fc1[hw_name] = pd.Series(hw1.fittedvalues.iloc[-H:].values, index=test.index)
print(f'alpha = {al:.3f} (level reacts quickly), gamma = {ga:.3f} (daily pattern is kept fixed)')
print('A:', {k: round(v, 2) for k, v in metrics(test, fc168[hw_name]).items()})
print('B:', {k: round(v, 2) for k, v in metrics(test, fc1[hw_name]).items()})

## 5. Time-series libraries: Prophet and TSFresh
**Prophet** (Meta) fits an additive model: trend + daily + weekly seasonality + holiday effects. We give it the Czech public holidays (planned in the synopsis). **TSFresh** extracts statistical features from time-series windows; here it is used in **preprocessing** to summarise each day into features and find unusual days.

In [ ]:
from prophet import Prophet
hdf = pd.DataFrame({'holiday': 'cz_holiday', 'ds': hol})
pdf = pd.DataFrame({'ds': train.index.tz_localize(None), 'y': train.values})
prophet = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, holidays=hdf)
prophet.fit(pdf)
pp = prophet.predict(pd.DataFrame({'ds': test.index.tz_localize(None)}))
fc168['Prophet'] = pd.Series(pp['yhat'].values, index=test.index)
print('A:', {k: round(v, 2) for k, v in metrics(test, fc168['Prophet']).items()})
fig = prophet.plot_components(prophet.predict(pdf.tail(24*14)[['ds']])); save('m2_02_prophet_components')

In [ ]:
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters
last = train.iloc[-24*90:]                                      # last 90 days
long = pd.DataFrame({'id': last.index.tz_localize(None).date.astype(str), 'time': last.index.hour, 'value': last.values})
feat = extract_features(long, column_id='id', column_sort='time', default_fc_parameters=MinimalFCParameters(),
                        disable_progressbar=True, n_jobs=0)
print('Daily features extracted by TSFresh:', [c.replace('value__', '') for c in feat.columns])
z = (feat['value__maximum'] - feat['value__maximum'].median()) / (1.4826 * (feat['value__maximum'] - feat['value__maximum'].median()).abs().median())
print('Most unusual days (daily peak far above a typical day):')
pd.DataFrame({'daily_max_GB/h': feat['value__maximum'], 'daily_mean': feat['value__mean'], 'robust_z': z}).loc[z.sort_values(ascending=False).head(5).index].round(1)

## 6. AutoTS – automatic search for the best-fitting model
AutoTS tries many models and pre-processing transformations, scores them on validation periods taken from the **end of the training data**, and returns the best (it scores with a mix of error metrics). We then judge its choice on our untouched test week like every other model. The search is kept small (7 light model families, 2 generations, 1 worker) so it fits in Colab memory and runs in a few minutes.

In [ ]:
from autots import AutoTS
atdf = pd.DataFrame({'datetime': train.index.tz_localize(None), 'GB': train.values})
t0 = time.time()
at = AutoTS(forecast_length=H, frequency='h', prediction_interval=0.95, ensemble=None,
            model_list=['LastValueNaive', 'AverageValueNaive', 'SeasonalNaive', 'ETS'],
            transformer_list='superfast', max_generations=0, num_validations=1, validation_method='backwards', n_jobs=1, verbose=0)
at = at.fit(atdf, date_col='datetime', value_col='GB')
autots_name = 'AutoTS (' + at.best_model_name + ')'
fc168[autots_name] = pd.Series(at.predict().forecast.iloc[:, 0].values, index=test.index)
print(f'AutoTS finished in {time.time()-t0:.0f}s | best model chosen: {at.best_model_name}')
print('A:', {k: round(v, 2) for k, v in metrics(test, fc168[autots_name]).items()})
at.results()[['Model', 'smape', 'mae', 'rmse']].head(5)

## 7. Comparison of all approaches – table and plots
Simple reference forecasts are added: **naive** (repeat the last value), **seasonal naive 24 h** (same hour yesterday) and **seasonal naive 168 h** (same hour last week). A model is only useful if it beats these.

In [ ]:
fc168['Naive']                 = pd.Series(train.iloc[-1], index=test.index)
fc168['Seasonal naive (24h)']  = pd.Series(np.tile(train.iloc[-24:].values, H // 24), index=test.index)
fc168['Seasonal naive (168h)'] = pd.Series(train.iloc[-168:].values, index=test.index)
fc1['Naive']                   = pd.Series(full.shift(1).iloc[-H:].values, index=test.index)
fc1['Seasonal naive (24h)']    = pd.Series(full.shift(24).iloc[-H:].values, index=test.index)
fc1['Seasonal naive (168h)']   = pd.Series(full.shift(168).iloc[-H:].values, index=test.index)

rows = []
for name in fc168:
    a = metrics(test, fc168[name]); b = metrics(test, fc1[name]) if name in fc1 else dict(RMSE=np.nan, MAE=np.nan, MAPE=np.nan)
    rows.append({'Model': name, 'A RMSE': a['RMSE'], 'A MAE': a['MAE'], 'A MAPE %': a['MAPE'],
                 'B RMSE': b['RMSE'], 'B MAE': b['MAE'], 'B MAPE %': b['MAPE']})
res_df = pd.DataFrame(rows).set_index('Model').sort_values('A RMSE')
print('A = 7-day-ahead forecast | B = 1-hour-ahead forecast (blank = the method cannot do that task)')
res_df.round(2)

In [ ]:
best_A = res_df['A RMSE'].idxmin(); best_B = res_df['B RMSE'].idxmin()
print(f"Best 7-day-ahead model : {best_A} (RMSE {res_df.loc[best_A, 'A RMSE']:.2f} GB/h)")
print(f"Best 1-hour-ahead model: {best_B} (RMSE {res_df.loc[best_B, 'B RMSE']:.2f} GB/h)")
below = res_df[res_df['B RMSE'] < 20].sort_values('B RMSE')
print('\nModels with RMSE below 20 GB/h (1-hour-ahead):', ', '.join(f'{i} ({v:.2f})' for i, v in below['B RMSE'].items()) if len(below) else 'none')

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
res_df['A RMSE'].sort_values(ascending=False).plot.barh(ax=ax[0], color='steelblue'); ax[0].set_title('Task A: 7-day-ahead RMSE (GB/h)'); ax[0].set_ylabel('')
res_df['B RMSE'].dropna().sort_values(ascending=False).plot.barh(ax=ax[1], color='darkorange')
ax[1].axvline(20, color='crimson', ls='--', lw=1.2); ax[1].set_title('Task B: 1-hour-ahead RMSE (dashed line = 20)'); ax[1].set_ylabel('')
save('m2_03_rmse_comparison')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(train.index[-48:], train.iloc[-48:], color='grey', lw=1, label='Train (last 2 days)')
ax.plot(test.index, test, color='black', lw=1.6, label='Actual')
for name in dict.fromkeys([best_A, sarima_name, hw_name, 'Prophet', arima_name]):
    ax.plot(test.index, fc168[name], lw=1.2, label=name)
ax.set_ylabel('GB per hour'); ax.set_title('Task A: 7-day-ahead forecasts vs actual'); ax.legend(ncol=2, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b'))
save('m2_04_forecast_A')

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(test.index, test, color='black', lw=1.4, label='Actual')
ax.plot(test.index, fc1[best_B], color='crimson', lw=1.2, label=f'{best_B} (1-hour-ahead)')
ax.set_ylabel('GB per hour'); ax.set_title('Task B: 1-hour-ahead forecast vs actual'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b'))
save('m2_05_forecast_B')

In [ ]:
# SARIMA 7-day forecast with its 95% prediction interval
m_s, fc_s = fits[sarima_name]; ci = fc_s.conf_int(alpha=0.05)
cover = ((test.values >= ci.iloc[:, 0].values) & (test.values <= ci.iloc[:, 1].values)).mean()
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.plot(test.index, test, color='black', lw=1.4, label='Actual')
ax.plot(test.index, fc168[sarima_name], color='crimson', lw=1.4, label='SARIMA forecast')
ax.fill_between(test.index, ci.iloc[:, 0], ci.iloc[:, 1], color='crimson', alpha=0.15, label='95% interval')
ax.set_title(f'SARIMA with 95% prediction interval (actual inside the band: {cover:.0%})'); ax.set_ylabel('GB per hour'); ax.legend()
save('m2_06_sarima_interval')

### 7.1 Why does error depend on the forecast horizon?
The same models are scored for forecasts made 1, 6, 24 and 168 hours ahead (always from the real data up to the forecast origin). Error grows with the horizon because the *known* daily pattern is weak in this traffic (seasonal strength 0.08) and the rest is unpredictable noise and bursts.

In [ ]:
ap = fits[sarima_name][0].apply(full)
def sar_h(h):  # forecasts of horizon h issued every h hours through the test week, from the real data so far
    return np.concatenate([ap.get_prediction(start=len(W)+s, end=len(W)+min(s+h, H)-1, dynamic=0).predicted_mean.values for s in range(0, H, h)])
def hw_h(h):
    out = []
    for s in range(0, H, h):
        f = ExponentialSmoothing(full.iloc[:len(W)+s], seasonal='add', seasonal_periods=24, initialization_method='known',
                                 initial_level=hw.params['initial_level'], initial_seasonal=hw.params['initial_seasons']
                                 ).fit(smoothing_level=al, smoothing_seasonal=ga, optimized=False)
        out.append(f.forecast(min(h, H-s)).values)
    return np.concatenate(out)
def naive_h(h): return np.concatenate([np.repeat(full.iloc[len(W)+s-1], min(h, H-s)) for s in range(0, H, h)])
hz = pd.DataFrame({h: {sarima_name: metrics(test, sar_h(h))['RMSE'], hw_name: metrics(test, hw_h(h))['RMSE'],
                       'Naive': metrics(test, naive_h(h))['RMSE']} for h in [1, 6, 24, 168]}).T
hz.index.name = 'horizon (h)'
print('RMSE (GB/h) by forecast horizon'); hz.round(2)

## 8. Residual diagnostics and burst alerts (objective from the synopsis)
A good model leaves residuals that look like random noise. **Burst alerts** use the same forecasts: an hour is flagged when the real traffic is far above what the model expected from the previous hours (robust z-score > 3, i.e. residual / (1.4826 × MAD)).

In [ ]:
from scipy import stats
mB = fits[sarima_name][0].apply(y)                       # one-step-ahead predictions for the whole series (parameters from W)
pred_all = mB.get_prediction().predicted_mean
res_all = (y - pred_all).iloc[24*7:]                     # skip the warm-up week
resid = res_all.iloc[-24*7*8:]
fig, ax = plt.subplots(2, 2, figsize=(13, 6))
ax[0,0].plot(resid.index, resid, lw=0.5); ax[0,0].axhline(0, color='k', lw=0.6); ax[0,0].set_title('One-step residuals (last 8 weeks)')
sns.histplot(resid, kde=True, ax=ax[0,1], color='steelblue'); ax[0,1].set_title('Residual histogram (heavy right tail = bursts)')
plot_acf(resid, lags=72, ax=ax[1,0]); ax[1,0].set_title('Residual ACF')
stats.probplot(resid, plot=ax[1,1]); ax[1,1].set_title('Q-Q plot')
save('m2_07_residuals')
lb = acorr_ljungbox(resid, lags=[24], return_df=True)
print(f'Residual mean {resid.mean():.2f} | std {resid.std():.2f} | Ljung-Box (24 lags) p = {lb.lb_pvalue.iloc[0]:.1e}')

In [ ]:
sigma = 1.4826 * (res_all - res_all.median()).abs().median()
zres = (res_all - res_all.median()) / sigma
alerts = zres[zres > 3]
q1, q3 = y.quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
iqr_hours = y[y > fence].index.intersection(res_all.index)
hit = alerts.index.intersection(iqr_hours)
print(f'Residual-based burst alerts: {len(alerts)} hours ({len(alerts)/len(res_all):.1%}) | IQR burst hours (Milestone 1): {len(iqr_hours)}')
print(f'Alerts that are also IQR bursts: {len(hit)} ({len(hit)/max(len(alerts),1):.0%} of alerts) | IQR bursts caught: {len(hit)/len(iqr_hours):.0%}')
top = (y.loc[alerts.index].to_frame('actual GB/h').assign(expected=pred_all.loc[alerts.index], z=zres.loc[alerts.index]).sort_values('z', ascending=False).head(5))
print('Five strongest alerts:'); print(top.round(1))
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(y.index, y, lw=0.4, color='steelblue', label='Hourly traffic')
ax.scatter(alerts.index, y.loc[alerts.index], s=10, color='crimson', label=f'burst alert ({len(alerts)})', zorder=3)
ax.set_title('Burst alerts from one-step SARIMA forecast residuals'); ax.set_ylabel('GB per hour'); ax.legend()
save('m2_08_burst_alerts')

**Reading the alerts.** Residual alerts mark *surprises*: hours much higher than the model expected from the previous hours. They overlap strongly with the burst hours found by the IQR rule in Milestone 1, but are not identical: a high hour that builds up gradually is predictable (no alert), while the first hours of a sudden burst are flagged at once. This is the forecast-based anomaly detection promised in the synopsis.

## 9. Robustness: is the ranking just luck of one test week?
The key models are re-fitted and re-scored on four different consecutive test weeks (each model sees only the data before its week).

In [ ]:
def eval_week(end):
    tr, te = y.iloc[:end], y.iloc[end:end+H]; Wk = tr.iloc[-24*7*8:]; out = {}
    out['Seasonal naive (168h)'] = metrics(te, tr.iloc[-168:].values)['RMSE']
    out[hw_name] = metrics(te, ExponentialSmoothing(Wk, seasonal='add', seasonal_periods=24, initialization_method='estimated').fit().forecast(H).values)['RMSE']
    out[arima_name] = metrics(te, SARIMAX(Wk, order=(pq_best[0],1,pq_best[1]), trend='n', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100).get_forecast(H).predicted_mean.values)['RMSE']
    out[sarima_name] = metrics(te, SARIMAX(Wk, order=s_order, seasonal_order=s_seasonal, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100).get_forecast(H).predicted_mean.values)['RMSE']
    return out
ends = [len(y) - H*k for k in (4, 3, 2, 1)]
roll = pd.DataFrame([eval_week(e) for e in ends], index=[f'week of {y.index[e]:%d %b}' for e in ends])
roll.loc['mean'] = roll.mean()
print('7-day-ahead RMSE (GB/h) on four different test weeks'); roll.round(2)

In [ ]:
print('SUMMARY')
print(f'Train {len(train)} pts | test {H} h | fitting window {len(W)} pts')
print(f'Models fitted: AR({p_best}), MA({q_best}), ARMA{pq_best}, ARIMA, SARIMA{s_order}{s_seasonal}, ARIMAX, SARIMAX, Holt-Winters, VARIMA, Prophet, AutoTS({at.best_model_name}) + 3 naive references')
print(f"Best 7-day-ahead : {best_A:34s} RMSE {res_df.loc[best_A,'A RMSE']:.2f} | MAE {res_df.loc[best_A,'A MAE']:.2f} | MAPE {res_df.loc[best_A,'A MAPE %']:.1f}%")
print(f"Best 1-hour-ahead: {best_B:34s} RMSE {res_df.loc[best_B,'B RMSE']:.2f} | MAE {res_df.loc[best_B,'B MAE']:.2f} | MAPE {res_df.loc[best_B,'B MAPE %']:.1f}%")
print(f'Test-week std = {test.std():.1f} GB/h (the 7-day error cannot fall far below this level for any model)')
res_df.round(3).to_csv('milestone2_results.csv'); shutil.make_archive('milestone2_figures', 'zip', 'figures')
from google.colab import files
files.download('milestone2_results.csv'); files.download('milestone2_figures.zip')


# Milestone 3 (Final): Machine-Learning and Deep-Learning Models, Comparison and Insights
Project: Forecasting and Burst Detection in Institutional Network Traffic (CESNET) | Target: n_bytes in GB per hour

| Phase 3 requirement | Where it is done |
|---|---|
| Implement and fine-tune LSTM / GRU, XGBoost, Prophet | Sections 2, 3, 4 (each tuned on a validation period) |
| Comparative analysis: classical vs ML/DL | Sections 1, 5, 6 |
| Actionable insights: alerts, patterns, forecast dashboard | Section 7 |
| Limitations, challenges, future scope | Section 8 |

Same fair test as Milestone 2: the last 7 days are never used for training or tuning.
* Task A - 7-day-ahead forecast (planning)
* Task B - 1-hour-ahead forecast (operations / alerts)
All hyper-parameters are chosen on a validation period taken from the end of the training data.

## 0. Setup

In [ ]:
!pip install -q prophet

In [ ]:
import logging, shutil, itertools
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
logging.getLogger('cmdstanpy').setLevel(logging.ERROR); logging.getLogger('prophet').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
SEED = 42; np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

H = 168
train, test = y.iloc[:-H], y.iloc[-H:]
n_tr = len(train)
W = train.iloc[-24*7*8:]; full = pd.concat([W, test])        # window for the classical baselines (same as Milestone 2)

def metrics(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float); e = a - p
    return dict(RMSE=np.sqrt(np.mean(e**2)), MAE=np.mean(np.abs(e)), MAPE=np.mean(np.abs(e / a)) * 100)

hol = pd.to_datetime(['2023-11-17','2023-12-25','2023-12-26','2024-01-01','2024-03-29','2024-04-01','2024-05-01','2024-05-08','2024-07-05'])
def calendar_basic(idx):
    loc = idx.tz_convert('Europe/Prague')
    return pd.DataFrame({'hour_sin': np.sin(2*np.pi*loc.hour/24), 'hour_cos': np.cos(2*np.pi*loc.hour/24),
                         'dow_sin': np.sin(2*np.pi*loc.dayofweek/7), 'dow_cos': np.cos(2*np.pi*loc.dayofweek/7),
                         'weekend': (loc.dayofweek >= 5).astype(float),
                         'holiday': loc.tz_localize(None).normalize().isin(hol).astype(float)}, index=idx)

A_fc, B_fc = {}, {}       # forecasts: Task A (7 days ahead) and Task B (1 hour ahead)
group = {}                # model type for the final table
print(f'Train {n_tr} pts | test {H} h | test-week std {test.std():.1f} GB/h | TensorFlow {tf.__version__} | XGBoost {xgb.__version__}')

## 1. Classical baselines (from Milestone 2) — SARIMA(2,0,2)(0,1,1)24, Holt-Winters and the naive references are re-fitted on the same 8-week window so the ML/DL models are compared on exactly the same test week.

In [ ]:
A_fc['Naive']                 = pd.Series(train.iloc[-1], index=test.index);            B_fc['Naive']                 = pd.Series(full.shift(1).iloc[-H:].values, index=test.index)
A_fc['Seasonal naive (168h)'] = pd.Series(train.iloc[-168:].values, index=test.index); B_fc['Seasonal naive (168h)'] = pd.Series(full.shift(168).iloc[-H:].values, index=test.index)

sar = SARIMAX(W, order=(2,0,2), seasonal_order=(0,1,1,24), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
A_fc['SARIMA'] = pd.Series(sar.get_forecast(H).predicted_mean.values, index=test.index)
B_fc['SARIMA'] = pd.Series(sar.apply(full).get_prediction(start=len(W)).predicted_mean.values, index=test.index)

hw = ExponentialSmoothing(W, seasonal='add', seasonal_periods=24, initialization_method='estimated').fit()
A_fc['Holt-Winters'] = pd.Series(hw.forecast(H).values, index=test.index)
hw1 = ExponentialSmoothing(full, seasonal='add', seasonal_periods=24, initialization_method='known', initial_level=hw.params['initial_level'],
                           initial_seasonal=hw.params['initial_seasons']).fit(smoothing_level=hw.params['smoothing_level'],
                                                                              smoothing_seasonal=hw.params['smoothing_seasonal'], optimized=False)
B_fc['Holt-Winters'] = pd.Series(hw1.fittedvalues.iloc[-H:].values, index=test.index)
for k in ['Naive', 'Seasonal naive (168h)', 'SARIMA', 'Holt-Winters']: group[k] = 'Classical'
for k in ['Naive', 'Seasonal naive (168h)', 'SARIMA', 'Holt-Winters']:
    print(f'{k:24s} A RMSE {metrics(test, A_fc[k])["RMSE"]:6.2f} | B RMSE {metrics(test, B_fc[k])["RMSE"]:6.2f}')

## 2. XGBoost - gradient-boosted trees on lag and calendar features — Task B: one row per hour from past values. Task A: a direct multi-horizon model (origin hour + horizon 1..168) using only information known at the origin plus the traffic at the same hour 1 to 4 weeks before the target. Settings tuned on a validation period.

In [ ]:
def make_features(s):
    d = pd.DataFrame({'y': s})
    for k in (1, 2, 3, 6, 12, 24, 48, 72, 168): d[f'lag{k}'] = s.shift(k)
    d['roll24']  = s.shift(1).rolling(24).mean();  d['std24'] = s.shift(1).rolling(24).std()
    d['roll168'] = s.shift(1).rolling(168).mean(); d['diff1'] = s.shift(1) - s.shift(2)
    return d.join(calendar_basic(s.index))
F = make_features(y).dropna()
featB = [c for c in F.columns if c != 'y']
print(f'{len(featB)} features for Task B:', featB)
F.head(3).round(2)

In [ ]:
def new_xgb(**p):
    return xgb.XGBRegressor(tree_method='hist', n_estimators=p.get('n', 300), max_depth=p.get('d', 4), learning_rate=p.get('lr', 0.05),
                            subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=2, random_state=SEED, n_jobs=-1, **p.get('extra', {}))

val_start = train.index[-336]
Ftr, Fva = F[F.index < val_start], F[(F.index >= val_start) & (F.index <= train.index[-1])]
gridB = [dict(d=d, lr=lr, n=n) for d in (3, 5) for lr in (0.03, 0.1) for n in (200, 500)]
resB = []
for p in gridB:
    m = new_xgb(**p).fit(Ftr[featB], Ftr['y'])
    resB.append({**p, 'val_RMSE': metrics(Fva['y'], m.predict(Fva[featB]))['RMSE']})
resB = pd.DataFrame(resB).sort_values('val_RMSE').reset_index(drop=True)
bestB = {k: int(v) if k in ('d', 'n') else float(v) for k, v in resB.iloc[0][['d', 'lr', 'n']].items()}
print('Best XGBoost settings for Task B (validation):', bestB)
resB.round(2)

In [ ]:
Ftrain = F[F.index <= train.index[-1]]
xgbB = new_xgb(**bestB).fit(Ftrain[featB], Ftrain['y'])
B_fc['XGBoost'] = pd.Series(xgbB.predict(F.loc[test.index, featB]), index=test.index)
group['XGBoost'] = 'ML / DL'
print('XGBoost Task B:', {k: round(v, 2) for k, v in metrics(test, B_fc['XGBoost']).items()})

In [ ]:
yv = y.values; r24 = y.rolling(24).mean().values; r168 = y.rolling(168).mean().values; sd168 = y.rolling(168).std().values
cal_all = calendar_basic(y.index).values
colsA = ['last', 'mean24', 'mean168', 'std168', 'same_hour_w1', 'same_hour_w2', 'same_hour_w3', 'same_hour_w4', 'same_hour_mean4', 'horizon',
         'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'weekend', 'holiday']
KS = np.arange(1, H + 1)
def build_A(origins):
    O = np.repeat(origins, H); K = np.tile(KS, len(origins)); T = O + K
    X = np.column_stack([yv[O], r24[O], r168[O], sd168[O], yv[T-168], yv[T-336], yv[T-504], yv[T-672],
                         (yv[T-168] + yv[T-336] + yv[T-504] + yv[T-672]) / 4, K, cal_all[T]])
    return pd.DataFrame(X, columns=colsA), yv[T]

o_first, step = 700, 6
val_o = n_tr - 1 - H                                       # validation week = the last 168 h of the training data
Xa_tr, ya_tr = build_A(np.arange(o_first, val_o - H + 1, step))
Xa_va, ya_va = build_A(np.array([val_o]))
gridA = [dict(d=d, lr=lr, n=300) for d in (4, 6) for lr in (0.05, 0.1)]
resA = []
for p in gridA:
    m = new_xgb(**p).fit(Xa_tr, ya_tr)
    resA.append({**p, 'val_RMSE': metrics(ya_va, m.predict(Xa_va))['RMSE']})
resA = pd.DataFrame(resA).sort_values('val_RMSE').reset_index(drop=True)
bestA = {k: int(v) if k in ('d', 'n') else float(v) for k, v in resA.iloc[0][['d', 'lr', 'n']].items()}
print(f'Task A training rows: {len(Xa_tr):,} | best XGBoost settings (validation week): {bestA}')
resA.round(2)

In [ ]:
Xa_full, ya_full = build_A(np.arange(o_first, n_tr - 1 - H + 1, step))
Xa_te, _ = build_A(np.array([n_tr - 1]))
xgbA = new_xgb(**bestA).fit(Xa_full, ya_full)
A_fc['XGBoost'] = pd.Series(xgbA.predict(Xa_te), index=test.index)
q_lo = new_xgb(**bestA, extra=dict(objective='reg:quantileerror', quantile_alpha=0.05)).fit(Xa_full, ya_full)
q_hi = new_xgb(**bestA, extra=dict(objective='reg:quantileerror', quantile_alpha=0.95)).fit(Xa_full, ya_full)
band_lo, band_hi = q_lo.predict(Xa_te), q_hi.predict(Xa_te)
cover = np.mean((test.values >= band_lo) & (test.values <= band_hi))
print('XGBoost Task A:', {k: round(v, 2) for k, v in metrics(test, A_fc['XGBoost']).items()})
print(f'90% band from quantile models: inside the band {cover:.0%} of test hours | average band width {np.mean(band_hi - band_lo):.0f} GB/h')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, m, cols, t in [(ax[0], xgbB, featB, 'Task B (1 hour ahead)'), (ax[1], xgbA, colsA, 'Task A (7 days ahead)')]:
    imp = pd.Series(m.feature_importances_, index=cols).sort_values().tail(10)
    imp.plot.barh(ax=a, color='steelblue'); a.set_title(f'XGBoost: top features, {t}')
save('m3_01_xgb_importance')

## 3. Deep learning - LSTM and GRU — recurrent networks read the traffic hour by hour and keep a memory; scaling uses training data only. Task B: last 48 hours into an LSTM or GRU (units and dropout tuned on the last 15% of training, with early stopping). Task A: an LSTM sequence-to-sequence model reads the last 168 hours and outputs the next 168, using only the known calendar of those future hours.

In [ ]:
mu, sd = train.mean(), train.std()
ys = ((y - mu) / sd).values.astype('float32')
cal4 = calendar_basic(y.index)[['hour_sin', 'hour_cos', 'weekend', 'holiday']].values.astype('float32')
feat5 = np.column_stack([ys, cal4]).astype('float32')

L = 48
def seq_B(positions):
    return np.stack([feat5[p-L:p] for p in positions]), cal4[positions], ys[positions]
pos_all = np.arange(L, n_tr)
cut = int(len(pos_all) * 0.85)
Xs_tr, Xc_tr, y_tr = seq_B(pos_all[:cut]); Xs_va, Xc_va, y_va = seq_B(pos_all[cut:])
test_pos = np.arange(n_tr, n_tr + H)
Xs_te, Xc_te, _ = seq_B(test_pos)

def build_rnn(kind, units, dropout):
    s_in = keras.Input((L, 5)); c_in = keras.Input((4,))
    rnn = layers.LSTM(units) if kind == 'LSTM' else layers.GRU(units)
    h = layers.Dropout(dropout)(rnn(s_in))
    h = layers.Dense(16, activation='relu')(layers.Concatenate()([h, c_in]))
    m = keras.Model([s_in, c_in], layers.Dense(1)(h)); m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return m

early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
rnn_res, rnn_models = [], {}
t0 = time.time()
for kind in ('LSTM', 'GRU'):
    for units, drop in [(32, 0.1), (64, 0.3)]:
        keras.utils.set_random_seed(SEED)
        m = build_rnn(kind, units, drop)
        h = m.fit([Xs_tr, Xc_tr], y_tr, validation_data=([Xs_va, Xc_va], y_va), epochs=60, batch_size=64, callbacks=[early], verbose=0)
        rnn_res.append(dict(model=kind, units=units, dropout=drop, epochs=len(h.history['loss']), val_RMSE_GB=float(np.sqrt(min(h.history['val_loss'])) * sd)))
        rnn_models[(kind, units, drop)] = m
rnn_res = pd.DataFrame(rnn_res).sort_values('val_RMSE_GB').reset_index(drop=True)
print(f'Tuning finished in {time.time()-t0:.0f}s')
rnn_res.round(2)

In [ ]:
for kind in ('LSTM', 'GRU'):
    r = rnn_res[rnn_res.model == kind].iloc[0]
    m = rnn_models[(kind, int(r.units), float(r.dropout))]
    pred = m.predict([Xs_te, Xc_te], verbose=0).ravel() * sd + mu
    B_fc[kind] = pd.Series(pred, index=test.index); group[kind] = 'ML / DL'
    print(f'{kind} (units {int(r.units)}, dropout {r.dropout}) Task B:', {k: round(v, 2) for k, v in metrics(test, B_fc[kind]).items()})

In [ ]:
LA = 168
def seq_A(origins):
    enc = np.stack([feat5[o-LA+1:o+1] for o in origins]); dec = np.stack([cal4[o+1:o+1+H] for o in origins])
    tgt = np.stack([ys[o+1:o+1+H] for o in origins])[..., None]
    return enc, dec, tgt
orig = np.arange(LA - 1, n_tr - 1 - H + 1, 2)
cutA = int(len(orig) * 0.85)
enc_tr, dec_tr, tg_tr = seq_A(orig[:cutA]); enc_va, dec_va, tg_va = seq_A(orig[cutA:])
enc_te, dec_te, _ = np.stack([feat5[n_tr-LA:n_tr]]), np.stack([cal4[n_tr:n_tr+H]]), None

def build_s2s(units=48, dropout=0.2):
    e_in = keras.Input((LA, 5)); d_in = keras.Input((H, 4))
    _, h, c = layers.LSTM(units, return_state=True)(e_in)
    dec = layers.LSTM(units, return_sequences=True)(d_in, initial_state=[h, c])
    out = layers.TimeDistributed(layers.Dense(1))(layers.Dropout(dropout)(dec))
    m = keras.Model([e_in, d_in], out); m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse'); return m

keras.utils.set_random_seed(SEED)
s2s = build_s2s()
t0 = time.time()
hist = s2s.fit([enc_tr, dec_tr], tg_tr, validation_data=([enc_va, dec_va], tg_va), epochs=40, batch_size=32,
               callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)], verbose=0)
A_fc['LSTM seq2seq'] = pd.Series(s2s.predict([enc_te, dec_te], verbose=0).ravel() * sd + mu, index=test.index); group['LSTM seq2seq'] = 'ML / DL'
print(f'Seq2seq trained {len(hist.history["loss"])} epochs in {time.time()-t0:.0f}s | validation RMSE {np.sqrt(min(hist.history["val_loss"]))*sd:.1f} GB/h')
print('LSTM seq2seq Task A:', {k: round(v, 2) for k, v in metrics(test, A_fc['LSTM seq2seq']).items()})
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(hist.history['loss'], label='train'); ax[0].plot(hist.history['val_loss'], label='validation'); ax[0].set_title('LSTM seq2seq learning curve (scaled MSE)'); ax[0].legend()
ax[1].plot(test.index, test, color='black', lw=1.2, label='Actual'); ax[1].plot(test.index, A_fc['LSTM seq2seq'], color='crimson', label='LSTM seq2seq'); ax[1].set_title('7-day forecast'); ax[1].legend()
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
save('m3_02_lstm_seq2seq')

## 4. Prophet - tuned — tuned on the validation week: trend flexibility (changepoint_prior_scale) and additive vs multiplicative seasonality.

In [ ]:
from prophet import Prophet
hdf = pd.DataFrame({'holiday': 'cz_holiday', 'ds': hol})
def prophet_fc(tr, idx, cps, mode):
    pm = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, holidays=hdf,
                 changepoint_prior_scale=cps, seasonality_mode=mode)
    pm.fit(pd.DataFrame({'ds': tr.index.tz_localize(None), 'y': tr.values}))
    return pm.predict(pd.DataFrame({'ds': idx.tz_localize(None)}))['yhat'].values
tr_v, va_v = train.iloc[:-H], train.iloc[-H:]
resP = pd.DataFrame([{'changepoint_prior_scale': c, 'seasonality_mode': m, 'val_RMSE': metrics(va_v, prophet_fc(tr_v, va_v.index, c, m))['RMSE']}
                     for c in (0.01, 0.05, 0.5) for m in ('additive', 'multiplicative')]).sort_values('val_RMSE').reset_index(drop=True)
bp = resP.iloc[0]
A_fc['Prophet (tuned)'] = pd.Series(prophet_fc(train, test.index, bp.changepoint_prior_scale, bp.seasonality_mode), index=test.index); group['Prophet (tuned)'] = 'ML / DL'
print('Best Prophet settings (validation):', dict(bp[['changepoint_prior_scale', 'seasonality_mode']]))
print('Prophet Task A:', {k: round(v, 2) for k, v in metrics(test, A_fc['Prophet (tuned)']).items()})
resP.round(2)

## 5. Ensemble — averaging forecasts from different model families often cancels their individual mistakes; the ensemble is fixed in advance (no test-week tuning): classical SARIMA / Holt-Winters, XGBoost and the LSTM.

In [ ]:
need_A = ['SARIMA', 'XGBoost', 'LSTM seq2seq']
need_B = ['Holt-Winters', 'XGBoost', 'LSTM']
print('Task A forecasts present:', list(A_fc))
print('Task B forecasts present:', list(B_fc))
print('Missing for A:', [k for k in need_A if k not in A_fc], '| Missing for B:', [k for k in need_B if k not in B_fc])

In [ ]:
# ===== XGBoost (Task B) =====
def make_features(s):
    d = pd.DataFrame({'y': s})
    for k in (1, 2, 3, 6, 12, 24, 48, 72, 168): d[f'lag{k}'] = s.shift(k)
    d['roll24']  = s.shift(1).rolling(24).mean();  d['std24'] = s.shift(1).rolling(24).std()
    d['roll168'] = s.shift(1).rolling(168).mean(); d['diff1'] = s.shift(1) - s.shift(2)
    return d.join(calendar_basic(s.index))
F = make_features(y).dropna()
featB = [c for c in F.columns if c != 'y']

def new_xgb(**p):
    return xgb.XGBRegressor(tree_method='hist', n_estimators=p.get('n', 300), max_depth=p.get('d', 4), learning_rate=p.get('lr', 0.05),
                            subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=2, random_state=SEED, n_jobs=-1, **p.get('extra', {}))

val_start = train.index[-336]
Ftr, Fva = F[F.index < val_start], F[(F.index >= val_start) & (F.index <= train.index[-1])]
gridB = [dict(d=d, lr=lr, n=n) for d in (3, 5) for lr in (0.03, 0.1) for n in (200, 500)]
resB = []
for p in gridB:
    m = new_xgb(**p).fit(Ftr[featB], Ftr['y'])
    resB.append({**p, 'val_RMSE': metrics(Fva['y'], m.predict(Fva[featB]))['RMSE']})
resB = pd.DataFrame(resB).sort_values('val_RMSE').reset_index(drop=True)
bestB = {k: int(v) if k in ('d', 'n') else float(v) for k, v in resB.iloc[0][['d', 'lr', 'n']].items()}
Ftrain = F[F.index <= train.index[-1]]
xgbB = new_xgb(**bestB).fit(Ftrain[featB], Ftrain['y'])
B_fc['XGBoost'] = pd.Series(xgbB.predict(F.loc[test.index, featB]), index=test.index); group['XGBoost'] = 'ML / DL'
print('XGBoost B | best settings', bestB, {k: round(v, 2) for k, v in metrics(test, B_fc['XGBoost']).items()})

# ===== XGBoost (Task A) =====
yv = y.values; r24 = y.rolling(24).mean().values; r168 = y.rolling(168).mean().values; sd168 = y.rolling(168).std().values
cal_all = calendar_basic(y.index).values
colsA = ['last', 'mean24', 'mean168', 'std168', 'same_hour_w1', 'same_hour_w2', 'same_hour_w3', 'same_hour_w4', 'same_hour_mean4', 'horizon',
         'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'weekend', 'holiday']
KS = np.arange(1, H + 1)
def build_A(origins):
    O = np.repeat(origins, H); K = np.tile(KS, len(origins)); T = O + K
    X = np.column_stack([yv[O], r24[O], r168[O], sd168[O], yv[T-168], yv[T-336], yv[T-504], yv[T-672],
                         (yv[T-168] + yv[T-336] + yv[T-504] + yv[T-672]) / 4, K, cal_all[T]])
    return pd.DataFrame(X, columns=colsA), yv[T]
o_first, step = 700, 6
val_o = n_tr - 1 - H
Xa_tr, ya_tr = build_A(np.arange(o_first, val_o - H + 1, step))
Xa_va, ya_va = build_A(np.array([val_o]))
gridA = [dict(d=d, lr=lr, n=300) for d in (4, 6) for lr in (0.05, 0.1)]
resA = []
for p in gridA:
    m = new_xgb(**p).fit(Xa_tr, ya_tr)
    resA.append({**p, 'val_RMSE': metrics(ya_va, m.predict(Xa_va))['RMSE']})
resA = pd.DataFrame(resA).sort_values('val_RMSE').reset_index(drop=True)
bestA = {k: int(v) if k in ('d', 'n') else float(v) for k, v in resA.iloc[0][['d', 'lr', 'n']].items()}
Xa_full, ya_full = build_A(np.arange(o_first, n_tr - 1 - H + 1, step))
Xa_te, _ = build_A(np.array([n_tr - 1]))
xgbA = new_xgb(**bestA).fit(Xa_full, ya_full)
A_fc['XGBoost'] = pd.Series(xgbA.predict(Xa_te), index=test.index)
q_lo = new_xgb(**bestA, extra=dict(objective='reg:quantileerror', quantile_alpha=0.05)).fit(Xa_full, ya_full)
q_hi = new_xgb(**bestA, extra=dict(objective='reg:quantileerror', quantile_alpha=0.95)).fit(Xa_full, ya_full)
band_lo, band_hi = q_lo.predict(Xa_te), q_hi.predict(Xa_te)
cover = np.mean((test.values >= band_lo) & (test.values <= band_hi))
print('XGBoost A | best settings', bestA, {k: round(v, 2) for k, v in metrics(test, A_fc['XGBoost']).items()}, f'| 90% band covers {cover:.0%}')

# ===== LSTM / GRU (Task B) =====
mu, sd = train.mean(), train.std()
ys = ((y - mu) / sd).values.astype('float32')
cal4 = calendar_basic(y.index)[['hour_sin', 'hour_cos', 'weekend', 'holiday']].values.astype('float32')
feat5 = np.column_stack([ys, cal4]).astype('float32')
L = 48
def seq_B(positions):
    return np.stack([feat5[p-L:p] for p in positions]), cal4[positions], ys[positions]
pos_all = np.arange(L, n_tr)
cut = int(len(pos_all) * 0.85)
Xs_tr, Xc_tr, y_tr = seq_B(pos_all[:cut]); Xs_va, Xc_va, y_va = seq_B(pos_all[cut:])
test_pos = np.arange(n_tr, n_tr + H)
Xs_te, Xc_te, _ = seq_B(test_pos)

def build_rnn(kind, units, dropout):
    s_in = keras.Input((L, 5)); c_in = keras.Input((4,))
    rnn = layers.LSTM(units) if kind == 'LSTM' else layers.GRU(units)
    h = layers.Dropout(dropout)(rnn(s_in))
    h = layers.Dense(16, activation='relu')(layers.Concatenate()([h, c_in]))
    m = keras.Model([s_in, c_in], layers.Dense(1)(h)); m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return m

early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
rnn_res, rnn_models = [], {}
t0 = time.time()
for kind in ('LSTM', 'GRU'):
    for units, drop in [(32, 0.1), (64, 0.3)]:
        keras.utils.set_random_seed(SEED)
        m = build_rnn(kind, units, drop)
        h = m.fit([Xs_tr, Xc_tr], y_tr, validation_data=([Xs_va, Xc_va], y_va), epochs=60, batch_size=64, callbacks=[early], verbose=0)
        rnn_res.append(dict(model=kind, units=units, dropout=drop, epochs=len(h.history['loss']), val_RMSE_GB=float(np.sqrt(min(h.history['val_loss'])) * sd)))
        rnn_models[(kind, units, drop)] = m
rnn_res = pd.DataFrame(rnn_res).sort_values('val_RMSE_GB').reset_index(drop=True)
print(f'RNN tuning finished in {time.time()-t0:.0f}s'); print(rnn_res.round(2))
for kind in ('LSTM', 'GRU'):
    r = rnn_res[rnn_res.model == kind].iloc[0]
    m = rnn_models[(kind, int(r.units), float(r.dropout))]
    B_fc[kind] = pd.Series(m.predict([Xs_te, Xc_te], verbose=0).ravel() * sd + mu, index=test.index); group[kind] = 'ML / DL'
    print(f'{kind} (units {int(r.units)}, dropout {r.dropout}) B:', {k: round(v, 2) for k, v in metrics(test, B_fc[kind]).items()})

# ===== LSTM seq2seq (Task A) =====
LA = 168
def seq_A(origins):
    enc = np.stack([feat5[o-LA+1:o+1] for o in origins]); dec = np.stack([cal4[o+1:o+1+H] for o in origins])
    tgt = np.stack([ys[o+1:o+1+H] for o in origins])[..., None]
    return enc, dec, tgt
orig = np.arange(LA - 1, n_tr - 1 - H + 1, 2)
cutA = int(len(orig) * 0.85)
enc_tr, dec_tr, tg_tr = seq_A(orig[:cutA]); enc_va, dec_va, tg_va = seq_A(orig[cutA:])
enc_te, dec_te = np.stack([feat5[n_tr-LA:n_tr]]), np.stack([cal4[n_tr:n_tr+H]])
def build_s2s(units=48, dropout=0.2):
    e_in = keras.Input((LA, 5)); d_in = keras.Input((H, 4))
    _, h, c = layers.LSTM(units, return_state=True)(e_in)
    dec = layers.LSTM(units, return_sequences=True)(d_in, initial_state=[h, c])
    out = layers.TimeDistributed(layers.Dense(1))(layers.Dropout(dropout)(dec))
    m = keras.Model([e_in, d_in], out); m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse'); return m
keras.utils.set_random_seed(SEED)
s2s = build_s2s(); t0 = time.time()
hist = s2s.fit([enc_tr, dec_tr], tg_tr, validation_data=([enc_va, dec_va], tg_va), epochs=40, batch_size=32,
               callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)], verbose=0)
A_fc['LSTM seq2seq'] = pd.Series(s2s.predict([enc_te, dec_te], verbose=0).ravel() * sd + mu, index=test.index); group['LSTM seq2seq'] = 'ML / DL'
print(f'Seq2seq trained {len(hist.history["loss"])} epochs in {time.time()-t0:.0f}s | A:', {k: round(v, 2) for k, v in metrics(test, A_fc['LSTM seq2seq']).items()})

# ===== Prophet (tuned) =====
from prophet import Prophet
hdf = pd.DataFrame({'holiday': 'cz_holiday', 'ds': hol})
def prophet_fc(tr, idx, cps, mode):
    pm = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, holidays=hdf,
                 changepoint_prior_scale=cps, seasonality_mode=mode)
    pm.fit(pd.DataFrame({'ds': tr.index.tz_localize(None), 'y': tr.values}))
    return pm.predict(pd.DataFrame({'ds': idx.tz_localize(None)}))['yhat'].values
tr_v, va_v = train.iloc[:-H], train.iloc[-H:]
resP = pd.DataFrame([{'changepoint_prior_scale': c, 'seasonality_mode': m, 'val_RMSE': metrics(va_v, prophet_fc(tr_v, va_v.index, c, m))['RMSE']}
                     for c in (0.01, 0.05, 0.5) for m in ('additive', 'multiplicative')]).sort_values('val_RMSE').reset_index(drop=True)
bp = resP.iloc[0]
A_fc['Prophet (tuned)'] = pd.Series(prophet_fc(train, test.index, bp.changepoint_prior_scale, bp.seasonality_mode), index=test.index); group['Prophet (tuned)'] = 'ML / DL'
print('Prophet best settings:', dict(bp[['changepoint_prior_scale', 'seasonality_mode']]), '| A:', {k: round(v, 2) for k, v in metrics(test, A_fc['Prophet (tuned)']).items()})

# ===== Ensemble =====
A_fc['Ensemble (SARIMA + XGBoost + LSTM)'] = (A_fc['SARIMA'] + A_fc['XGBoost'] + A_fc['LSTM seq2seq']) / 3
B_fc['Ensemble (HW + XGBoost + LSTM)']     = (B_fc['Holt-Winters'] + B_fc['XGBoost'] + B_fc['LSTM']) / 3
group['Ensemble (SARIMA + XGBoost + LSTM)'] = group['Ensemble (HW + XGBoost + LSTM)'] = 'Ensemble'
print('Ensemble A:', {k: round(v, 2) for k, v in metrics(test, A_fc['Ensemble (SARIMA + XGBoost + LSTM)']).items()})
print('Ensemble B:', {k: round(v, 2) for k, v in metrics(test, B_fc['Ensemble (HW + XGBoost + LSTM)']).items()})

## 6. Comparative analysis: classical vs ML / DL

In [ ]:
rows = []
for name in dict.fromkeys(list(A_fc) + list(B_fc)):
    a = metrics(test, A_fc[name]) if name in A_fc else dict(RMSE=np.nan, MAE=np.nan, MAPE=np.nan)
    b = metrics(test, B_fc[name]) if name in B_fc else dict(RMSE=np.nan, MAE=np.nan, MAPE=np.nan)
    rows.append({'Model': name, 'Type': group.get(name, 'Classical'), 'A RMSE': a['RMSE'], 'A MAE': a['MAE'], 'A MAPE %': a['MAPE'],
                 'B RMSE': b['RMSE'], 'B MAE': b['MAE'], 'B MAPE %': b['MAPE']})
res3 = pd.DataFrame(rows).set_index('Model').sort_values('A RMSE')
print('A = 7-day-ahead | B = 1-hour-ahead  (blank = the method cannot do that task)')
res3.round(2)

In [ ]:
bA, bB = res3['A RMSE'].idxmin(), res3['B RMSE'].idxmin()
print(f"Best 7-day-ahead : {bA} (RMSE {res3.loc[bA,'A RMSE']:.2f})")
print(f"Best 1-hour-ahead: {bB} (RMSE {res3.loc[bB,'B RMSE']:.2f})")
print('\nBest of each family, Task A:', res3.groupby('Type')['A RMSE'].min().round(2).to_dict())
print('Best of each family, Task B:', res3.groupby('Type')['B RMSE'].min().round(2).to_dict())
below = res3[res3['B RMSE'] < 20].sort_values('B RMSE')['B RMSE'].round(2)
print('\nModels with RMSE below 20 GB/h (1-hour-ahead):', dict(below) if len(below) else 'none')
colors = {'Classical': 'steelblue', 'ML / DL': 'darkorange', 'Ensemble': 'seagreen'}
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, col, t in [(ax[0], 'A RMSE', 'Task A: 7-day-ahead RMSE (GB/h)'), (ax[1], 'B RMSE', 'Task B: 1-hour-ahead RMSE (GB/h)')]:
    d = res3[col].dropna().sort_values(ascending=False)
    a.barh(d.index, d.values, color=[colors[res3.loc[i, 'Type']] for i in d.index]); a.set_title(t)
ax[1].axvline(20, color='crimson', ls='--', lw=1.2)
from matplotlib.patches import Patch
ax[0].legend(handles=[Patch(color=c, label=k) for k, c in colors.items()], loc='lower right')
save('m3_03_comparison')

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 8.5))
ax[0].plot(train.index[-48:], train.iloc[-48:], color='grey', lw=1, label='Train (last 2 days)'); ax[0].plot(test.index, test, color='black', lw=1.6, label='Actual')
for n in ['SARIMA', 'XGBoost', 'LSTM seq2seq', 'Prophet (tuned)', 'Ensemble (SARIMA + XGBoost + LSTM)']: ax[0].plot(test.index, A_fc[n], lw=1.2, label=n)
ax[0].set_title('Task A: 7-day-ahead forecasts vs actual'); ax[0].legend(ncol=3, fontsize=8); ax[0].set_ylabel('GB per hour')
ax[1].plot(test.index, test, color='black', lw=1.4, label='Actual')
for n in ['Holt-Winters', 'XGBoost', 'LSTM', 'GRU']: ax[1].plot(test.index, B_fc[n], lw=1.1, label=n)
ax[1].set_title('Task B: 1-hour-ahead forecasts vs actual'); ax[1].legend(ncol=4, fontsize=8); ax[1].set_ylabel('GB per hour')
for a in ax: a.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b'))
save('m3_04_forecasts')

### 6.1 Is the ranking just luck of one test week?
XGBoost and the classical models are re-fitted on four different consecutive test weeks, each seeing only the data before its week (the LSTM is skipped because re-training it four times is slow). Treat it as a stability check.

In [ ]:
def week_scores(end):
    te = y.iloc[end:end+H]; tr = y.iloc[:end]; Wk = tr.iloc[-24*7*8:]; fullk = pd.concat([Wk, te]); out = {}
    out['Seasonal naive (168h) A'] = metrics(te, tr.iloc[-168:].values)['RMSE']
    sk = SARIMAX(Wk, order=(2,0,2), seasonal_order=(0,1,1,24), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
    out['SARIMA A'] = metrics(te, sk.get_forecast(H).predicted_mean.values)['RMSE']
    out['SARIMA B'] = metrics(te, sk.apply(fullk).get_prediction(start=len(Wk)).predicted_mean.values)['RMSE']
    Fk = F[F.index < y.index[end]]; mk = new_xgb(**bestB).fit(Fk[featB], Fk['y'])
    out['XGBoost B'] = metrics(te, mk.predict(F.loc[te.index, featB]))['RMSE']
    Xk, yk = build_A(np.arange(o_first, end - 1 - H + 1, step)); ak = new_xgb(**bestA).fit(Xk, yk)
    out['XGBoost A'] = metrics(te, ak.predict(build_A(np.array([end - 1]))[0]))['RMSE']
    return out
ends = [len(y) - H*k for k in (4, 3, 2, 1)]
roll3 = pd.DataFrame([week_scores(e) for e in ends], index=[f'week of {y.index[e]:%d %b}' for e in ends])
roll3.loc['mean'] = roll3.mean()
print('RMSE (GB/h) on four different test weeks: A = 7-day-ahead, B = 1-hour-ahead')
roll3[['Seasonal naive (168h) A', 'SARIMA A', 'XGBoost A', 'SARIMA B', 'XGBoost B']].round(2)

## 7. Actionable insights for the network team
Three things the team can use: (1) what drives traffic, (2) burst alerts from the ML forecast, (3) a capacity plan with an uncertainty band.

### 7.1 Burst alerts (out-of-sample)
The one-hour-ahead XGBoost forecast is made with time-series cross-validation, so each block of hours is predicted by a model that saw only earlier data. An hour is flagged when it is more than 3 robust standard deviations above its forecast.

In [ ]:
tscv = TimeSeriesSplit(n_splits=6)
oof = pd.Series(np.nan, index=F.index)
for tr_i, te_i in tscv.split(F):
    mm = new_xgb(**bestB).fit(F.iloc[tr_i][featB], F.iloc[tr_i]['y'])
    oof.iloc[te_i] = mm.predict(F.iloc[te_i][featB])
oof = oof.dropna(); res_oof = F.loc[oof.index, 'y'] - oof
sig = 1.4826 * (res_oof - res_oof.median()).abs().median()
z = (res_oof - res_oof.median()) / sig
alerts = z[z > 3]
q1, q3 = y.quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
iqr_h = y[y > fence].index.intersection(oof.index); hit = alerts.index.intersection(iqr_h)
print(f'Out-of-sample period: {oof.index.min():%d %b %Y} -> {oof.index.max():%d %b %Y} ({len(oof)} hours)')
print(f'Burst alerts: {len(alerts)} ({len(alerts)/len(oof):.1%} of hours) | IQR burst hours in the same period: {len(iqr_h)}')
print(f'Alerts that are real IQR bursts (precision): {len(hit)/max(len(alerts),1):.0%} | IQR bursts caught (recall): {len(hit)/len(iqr_h):.0%}')
print('Five strongest alerts:')
print(pd.DataFrame({'actual GB/h': F.loc[alerts.index, 'y'], 'expected': oof.loc[alerts.index], 'z': z.loc[alerts.index]}).sort_values('z', ascending=False).head(5).round(1))
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(y.index, y, lw=0.4, color='steelblue', label='Hourly traffic'); ax.scatter(alerts.index, y.loc[alerts.index], s=10, color='crimson', label=f'burst alert ({len(alerts)})', zorder=3)
ax.set_title('Burst alerts from out-of-sample XGBoost forecasts'); ax.set_ylabel('GB per hour'); ax.legend()
save('m3_05_burst_alerts')

### 7.2 Patterns that drive traffic
The importance plot shows which signals the forecasts rely on; the table below turns the weekly rhythm into plain numbers.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, m, cols, t in [(ax[0], xgbB, featB, 'Task B (1 hour ahead)'), (ax[1], xgbA, colsA, 'Task A (7 days ahead)')]:
    imp = pd.Series(m.feature_importances_, index=cols).sort_values().tail(10)
    imp.plot.barh(ax=a, color='steelblue'); a.set_title(f'XGBoost: top features, {t}')
save('m3_01_xgb_importance')
yl = y.copy(); yl.index = yl.index.tz_convert('Europe/Prague')
hr = yl.groupby(yl.index.hour).mean(); dw = yl.groupby(yl.index.dayofweek).mean()
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
imp_B = pd.Series(xgbB.feature_importances_, index=featB).sort_values(ascending=False)
print('Top drivers of the next-hour forecast:', ', '.join(f'{k} ({v:.0%})' for k, v in imp_B.head(4).items()))
print(f'Quietest hour {hr.idxmin()}:00 ({hr.min():.0f} GB/h) | busiest hour {hr.idxmax()}:00 ({hr.max():.0f} GB/h)')
print(f'Quietest day {day_names[int(dw.idxmin())]} ({dw.min():.0f} GB/h) | busiest day {day_names[int(dw.idxmax())]} ({dw.max():.0f} GB/h)')

### 7.3 Forecast dashboard and capacity plan
One page for the operator: the next-week forecast with its 90% band, the latest alerts, the weekly pattern and key numbers.

In [ ]:
best_model_A = 'Ensemble (SARIMA + XGBoost + LSTM)' if res3.loc['Ensemble (SARIMA + XGBoost + LSTM)', 'A RMSE'] <= res3['A RMSE'].min() else bA
fig = plt.figure(figsize=(14, 8.5)); gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1])
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(train.index[-72:], train.iloc[-72:], color='grey', lw=1.2, label='Last 3 days (known)')
ax1.plot(test.index, test, color='black', lw=1.3, label='Actual (kept hidden from the model)')
ax1.plot(test.index, A_fc[best_model_A], color='crimson', lw=1.6, label=f'Forecast: {best_model_A}')
ax1.fill_between(test.index, band_lo, band_hi, color='crimson', alpha=0.12, label=f'90% band (XGBoost quantiles, {cover:.0%} inside)')
ax1.set_title('Next 7 days: forecast and uncertainty'); ax1.set_ylabel('GB per hour'); ax1.legend(ncol=2, fontsize=8); ax1.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b'))
ax2 = fig.add_subplot(gs[1, 0])
recent = y.iloc[-24*14:]; ar = alerts[alerts.index >= recent.index[0]]
ax2.plot(recent.index, recent, lw=0.8, color='steelblue'); ax2.scatter(ar.index, y.loc[ar.index], color='crimson', s=22, zorder=3, label=f'alerts ({len(ar)})')
ax2.set_title('Last 14 days: burst alerts'); ax2.legend(); ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax3 = fig.add_subplot(gs[1, 1]); heat = yl.groupby([yl.index.dayofweek, yl.index.hour]).mean().unstack(); heat.index = day_names
sns.heatmap(heat, cmap='YlOrRd', ax=ax3, cbar_kws={'label': 'GB/h'}); ax3.set_title('Typical week: weekday x hour (Prague time)'); ax3.set_xlabel('hour'); ax3.set_ylabel('')
save('m3_06_dashboard')
peak_fc, peak_up = A_fc[best_model_A].max(), band_hi.max()
print(f'CAPACITY PLAN for the week: expected peak {peak_fc:.0f} GB/h; plan for up to {peak_up:.0f} GB/h (upper 90% band). Actual peak that week: {test.max():.0f} GB/h.')
print(f'Typical hour: {test.mean():.0f} GB/h. Reserve headroom for bursts: the 99th percentile of history is {y.quantile(.99):.0f} GB/h.')

## 8. Interpretation, limitations, challenges and future scope

**What the comparison shows** (read it together with the table in Section 6).
* For the next hour, every model that uses recent values (XGBoost, LSTM, GRU, Holt-Winters, SARIMA and the ensemble) lands close together, all below 20 GB/h and well below the naive reference. The ensemble is best, but the flexible ML/DL models do not clearly beat a well-chosen classical model: with strong randomness and only 40 weeks of data, extra complexity buys little.
* For the week ahead, XGBoost is best on the final test week, but over four different weeks SARIMA is slightly better on average and XGBoost is steadier. The deep sequence-to-sequence model and Prophet are weaker because they have too little data to learn a reliable weekly pattern. Every model is limited by the traffic's own randomness (the test week varies by about 26 GB/h around its mean), and none can predict a burst before it starts.
* The 90% band covers most hours but not the week's highest peak, so capacity planning should also keep headroom based on the historical 99th percentile.
* The most useful result for the stakeholder is the system: a one-hour-ahead forecast that raises burst alerts (about three quarters of the alerts are real bursts) and a week-ahead forecast with an uncertainty band for capacity planning.

**Limitations.** One institution; about 40 weeks (no full yearly cycle); heavy bursts and level shifts that no model can foresee; the 7-day test week is one sample (checked on four weeks only for XGBoost and classical models); deep models are sensitive to the random seed and need more data; hyper-parameter grids were small to keep the notebook fast.

**Challenges.** Skewed, bursty data (skewness 2.57), one missing hour (interpolated), conflicting stationarity tests (handled by differencing), preventing data leakage (lags and calendar only, validation before test, scaling on training data), and keeping Task A honest by using only information known at the forecast origin.

**Future scope.** More institutions and a full year of data; exogenous signals (packets, flows, ASN counts) as lagged features; probabilistic deep models (quantile LSTM, temporal fusion transformer); online learning; a live dashboard with automatic alert notification.

In [ ]:
print('FINAL SUMMARY')
print(f'Test week std = {test.std():.1f} GB/h')
print(f"Best 7-day-ahead : {bA:36s} RMSE {res3.loc[bA,'A RMSE']:.2f} | MAE {res3.loc[bA,'A MAE']:.2f} | MAPE {res3.loc[bA,'A MAPE %']:.1f}%")
print(f"Best 1-hour-ahead: {bB:36s} RMSE {res3.loc[bB,'B RMSE']:.2f} | MAE {res3.loc[bB,'B MAE']:.2f} | MAPE {res3.loc[bB,'B MAPE %']:.1f}%")
print('Best per family, A:', res3.groupby('Type')['A RMSE'].min().round(2).to_dict(), '| B:', res3.groupby('Type')['B RMSE'].min().round(2).to_dict())
print(f'Burst alerts (out-of-sample XGBoost): {len(alerts)} | precision {len(hit)/max(len(alerts),1):.0%} | recall vs IQR bursts {len(hit)/len(iqr_h):.0%}')
res3.round(3).to_csv('milestone3_results.csv'); roll3.round(3).to_csv('milestone3_four_week_check.csv')
shutil.make_archive('milestone3_figures', 'zip', 'figures')
from google.colab import files
files.download('milestone3_results.csv'); files.download('milestone3_four_week_check.csv'); files.download('milestone3_figures.zip')